# 00 - Install & Setup

This notebook installs dependencies, tells the pipeline where the repo/models/python modules live, and **writes the 10 wrapper modules into `/content/implementation`** using `%%writefile` cells.

After running this once you never have to paste code manually.


## 1. Install dependencies


In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python pandas numpy matplotlib pillow "transformers>=4.36"


## 2. Repo location

Set `REPO_ROOT` to where the traffIQ repo is on Colab (after `git clone` or Drive mount).


In [ ]:
import os
REPO_ROOT = "/content/traffIQ"        # after:  !git clone <repo-url> /content/traffIQ
# REPO_ROOT = "/content/drive/MyDrive/traffIQ"   # after Drive mount

os.environ["TRAFFIQ_AI_DIR"] = f"{REPO_ROOT}/ai"
print("REPO_ROOT =", REPO_ROOT)
print("TRAFFIQ_AI_DIR =", os.environ["TRAFFIQ_AI_DIR"])


## 3. (Optional) upload model weights / demo clip

If the repo is already on Colab with `ai/models`, `ai/third_party/ref_repo/weights/`, and `ai/classes/`, skip this. Otherwise upload the five files listed in `docs/COLAB_SETUP.md` into a single folder and set `WEIGHTS_DIR` below.


In [ ]:
# from google.colab import files
# files.upload()   # pick best.pt, char.pt, lpsr_best.pth, object.pt, ocr_class_names.txt

# WEIGHTS_DIR = "/content/assets"   # if you uploaded them somewhere


## 4. Materialise the wrapper modules

The cells below write the adapter/pipeline modules. Do not edit them in the notebook — edit the real files under `implementation/` in the repo and rerun `python tools/make_notebooks.py`.


In [ ]:
!mkdir -p /content/implementation/adapters /content/implementation/pipeline
import sys; sys.path.insert(0, "/content/implementation")


In [ ]:
%%writefile /content/implementation/adapters/__init__.py


In [ ]:
%%writefile /content/implementation/adapters/vehicle_detector.py
import os
import sys
from typing import List

import cv2
import numpy as np
import torch

from pipeline.config import Config

VEHICLE_TYPES = {"car", "truck", "bus", "motorcycle", "motorbike"}


class VehicleDetector:
    """Wrapper around ref_repo object.pt (vendored yolov5, COCO-style classes)."""

    def __init__(self, cfg: Config, conf: float = None):
        self.cfg = cfg
        self.conf_thres = cfg.vehicle_conf if conf is None else conf
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self._model = self._load(cfg.vehicle_weights)

    def _import_detection(self):
        prev = os.getcwd()
        self.cfg.ensure_ai_on_path()
        try:
            if str(self.cfg.ref_repo_dir) not in sys.path:
                sys.path.insert(0, self.cfg.ref_repo_dir_str)
            os.chdir(self.cfg.ref_repo_dir_str)
            from my_models.detection import Detection

            return Detection
        finally:
            os.chdir(prev)

    def _load(self, weights_path):
        Detection = self._import_detection()
        return Detection(
            size=[self.cfg.vehicle_size, self.cfg.vehicle_size],
            weights_path=str(weights_path),
            device=self.device,
            iou_thres=self.cfg.vehicle_iou,
            conf_thres=self.conf_thres,
        )

    def detect(self, frame_bgr: np.ndarray) -> List[dict]:
        """Returns [{bbox:(x1,y1,x2,y2), type, type_confidence, conf}] for vehicle classes."""
        results, _ = self._model.detect(frame_bgr, bb_scale=True)
        out = []
        for name, conf_str, box in results:
            name = str(name).strip().lower()
            if name not in VEHICLE_TYPES:
                continue
            vals = [float(v) for v in box]
            if len(vals) != 4:
                continue
            x1, y1, x2, y2 = vals
            x1, x2 = sorted((max(0.0, x1), float(frame_bgr.shape[1])))
            y1, y2 = sorted((max(0.0, y1), float(frame_bgr.shape[0])))
            if x2 <= x1 or y2 <= y1:
                continue
            out.append({
                "bbox": (int(x1), int(y1), int(x2), int(y2)),
                "type": name,
                "type_confidence": float(conf_str),
                "conf": float(conf_str),
            })
        return out


In [ ]:
%%writefile /content/implementation/adapters/tracking_adapter.py
import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Union

from pipeline.config import Config


@dataclass
class TrackedBox:
    x1: int
    y1: int
    x2: int
    y2: int
    id: int
    cx: float = 0.0
    cy: float = 0.0
    type: str = "car"
    type_confidence: float = 0.0
    meta: dict = field(default_factory=dict)


Det = Union[Tuple[int, int, int, int], List, dict]


def _normalize_det(det: Det) -> Tuple[Tuple[int, int, int, int], dict]:
    if isinstance(det, dict):
        b = tuple(int(v) for v in det.get("bbox"))
        meta = dict(det)
        meta.pop("bbox", None)
        return b, meta
    b = tuple(int(v) for v in det[:4])
    return b, {}


class CenterPointTracker:
    """Center-point tracker (port of teammate's tracker.py) with configurable threshold."""

    def __init__(self, threshold: float = 35.0):
        self.threshold = float(threshold)
        self.center_points: Dict[int, Tuple[float, float]] = {}
        self.id_count = 0

    def reset(self):
        self.center_points.clear()
        self.id_count = 0

    def update(self, detections: List[Det]) -> List[TrackedBox]:
        tracked = []
        used_ids = set()
        for det in detections:
            (x1, y1, x2, y2), meta = _normalize_det(det)
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0
            matched_id = None
            for tid, (px, py) in self.center_points.items():
                d = math.hypot(cx - px, cy - py)
                if d < self.threshold:
                    matched_id = tid
                    break
            if matched_id is None:
                matched_id = self.id_count
                self.id_count += 1
            self.center_points[matched_id] = (cx, cy)
            used_ids.add(matched_id)
            tracked.append(TrackedBox(
                x1=x1, y1=y1, x2=x2, y2=y2, id=matched_id, cx=cx, cy=cy,
                type=str(meta.get("type", "car")),
                type_confidence=float(meta.get("type_confidence", meta.get("conf", 0.0))),
                meta=meta,
            ))
        for tid in [t for t in self.center_points if t not in used_ids]:
            del self.center_points[tid]
        return tracked


class ByteTrackAdapter:
    """Optional ByteTrack backend (ultralytics). Falls back to center tracker on any error."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._warned = False
        self._tracker = None
        try:
            from types import SimpleNamespace
            from ultralytics.trackers.byte_tracker import BYTETracker

            args = SimpleNamespace(
                track_buffer=cfg.bytetrack_buffer,
                match_thresh=cfg.bytetrack_match_thresh,
                min_box_area=cfg.bytetrack_min_box_area,
                mot20=False,
            )
            self._tracker = BYTETracker(args, frame_rate=30)
            self._fallback = CenterPointTracker(cfg.center_threshold)
            self._active = True
        except Exception as e:  # pragma: no cover - import env dependent
            print(f"[WARN] ByteTrack unavailable ({e}); using center-point tracker.")
            self._active = False
            self._fallback = CenterPointTracker(cfg.center_threshold)

    def update(self, detections: List[Det]) -> List[TrackedBox]:
        if not self._active:
            return self._fallback.update(detections)
        try:
            from types import SimpleNamespace

            tracks = []
            for det in detections:
                (x1, y1, x2, y2), meta = _normalize_det(det)
                tlbr = __import__("numpy").array([x1, y1, x2, y2], dtype=float)
                track = SimpleNamespace(tlbr=tlbr, conf=float(meta.get("conf", 0.9)),
                                        cls=int(meta.get("cls", 0)))
                tracks.append(track)
            out = self._tracker.update(tracks, None, None)
            tracked = []
            for t in out:
                tlbr = getattr(t, "tlbr", None)
                tid = getattr(t, "track_id", None)
                if tid is None:
                    tid = getattr(t, "id", None)
                if tlbr is None or tid is None:
                    continue
                x1, y1, x2, y2 = (int(v) for v in np_as_flat(tlbr))
                cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                tracked.append(TrackedBox(x1, y1, x2, y2, int(tid), cx, cy,
                                          "car", 0.0, {}))
            return tracked
        except Exception as e:
            if not self._warned:
                print(f"[WARN] ByteTrack update failed ({e}); using center-point tracker.")
                self._warned = True
            return self._fallback.update(detections)


def np_as_flat(arr):
    import numpy as np

    a = np.asarray(arr).reshape(-1)
    return a.tolist()


def get_tracker(cfg: Config) -> Union[CenterPointTracker, ByteTrackAdapter]:
    if cfg.tracker_kind == "bytetrack":
        return ByteTrackAdapter(cfg)
    return CenterPointTracker(cfg.center_threshold)


In [ ]:
%%writefile /content/implementation/adapters/speed_adapter.py
from typing import Dict, Optional

from pipeline.config import Config
from adapters.tracking_adapter import TrackedBox


class LineCrossingSpeedEstimator:
    """Two-virtual-line speed estimator (same method as Speed(up_down).ipynb).

    A tracked vehicle's centre records the video-time when it enters the band of the
    first line; crossing the second line yields elapsed time -> speed = distance / time.
    Returns one event per (track, direction) pass, with a re-arm cooldown.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.line_a_y = int(cfg.line_a_y)
        self.line_b_y = int(cfg.line_b_y)
        self.offset = int(cfg.line_offset)
        self.distance_m = float(cfg.distance_meters)
        self.rearm = int(cfg.rearm_frames)
        self.prune_missed = int(cfg.prune_missed)
        # pending[track_id] = (line_x1|line_x2, video_time, frame_index)
        self.pending: Dict[int, tuple] = {}
        self.last_frame_seen: Dict[int, int] = {}
        self.last_event_frame: Dict[int, int] = {}

    def reset(self):
        self.pending.clear()
        self.last_frame_seen.clear()
        self.last_event_frame.clear()

    def in_band(self, y: float, line_y: int) -> bool:
        return abs(y - line_y) <= self.offset

    def update(self, track: TrackedBox, frame_index: int, fps: float) -> Optional[dict]:
        fps = float(fps) if fps and fps > 0 else 20.0
        t = frame_index / fps
        self.last_frame_seen[track.id] = frame_index

        hit_a = self.in_band(track.cy, self.line_a_y)
        hit_b = self.in_band(track.cy, self.line_b_y)
        if hit_a == hit_b:
            self._prune(frame_index)
            return None

        pending = self.pending.get(track.id)
        if hit_b and pending is not None and pending[0] == "a":
            self.pending.pop(track.id, None)
            return self._emit(track, "ab", pending[1], t, frame_index)
        if hit_a and pending is not None and pending[0] == "b":
            self.pending.pop(track.id, None)
            return self._emit(track, "ba", pending[1], t, frame_index)

        if hit_a and pending is None:
            if frame_index - self.last_event_frame.get(track.id, -1e9) >= self.rearm:
                self.pending[track.id] = ("a", t, frame_index)
        elif hit_b and pending is None:
            if frame_index - self.last_event_frame.get(track.id, -1e9) >= self.rearm:
                self.pending[track.id] = ("b", t, frame_index)

        self._prune(frame_index)
        return None

    def _emit(self, track, direction_key, start_time, end_time, frame_index) -> dict:
        elapsed = end_time - start_time
        if elapsed <= 0:
            return None
        speed_ms = self.distance_m / elapsed
        speed_kmh = speed_ms * 3.6
        if direction_key == "ab":
            direction = self.cfg.direction_ab
        else:
            direction = self.cfg.direction_ba
        self.last_event_frame[track.id] = frame_index
        return {
            "value_kmh": float(speed_kmh),
            "estimated": bool(self.cfg.speed_estimated),
            "direction": direction,
            "elapsed_s": float(elapsed),
        }

    def _prune(self, frame_index: int):
        stale = [tid for tid, f in self.last_frame_seen.items()
                 if frame_index - f > self.prune_missed]
        for tid in stale:
            self.pending.pop(tid, None)
            self.last_frame_seen.pop(tid, None)
            self.last_event_frame.pop(tid, None)


In [ ]:
%%writefile /content/implementation/adapters/plate_detector.py
from typing import List

import cv2
import numpy as np

from pipeline.config import Config


class PlateDetector:
    """Wrapper around best.pt (ultralytics YOLO26m, class number_plate)."""

    def __init__(self, cfg: Config, conf: float = None):
        from ultralytics import YOLO

        self.cfg = cfg
        self.conf = cfg.plate_det_conf if conf is None else conf
        self.model = YOLO(str(cfg.plate_weights))
        self.names = self.model.names

    def detect(self, frame_bgr: np.ndarray) -> List[dict]:
        results = self.model.predict(frame_bgr, conf=self.conf, verbose=False)
        out = []
        for r in results:
            for b in r.boxes:
                x1, y1, x2, y2 = (int(v) for v in b.xyxy[0].tolist())
                conf = float(b.conf[0])
                out.append({"bbox": (x1, y1, x2, y2), "confidence": conf})
        return out

    def crop(self, frame_bgr: np.ndarray, bbox) -> np.ndarray:
        h, w = frame_bgr.shape[:2]
        x1, y1, x2, y2 = bbox
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        return frame_bgr[y1:y2, x1:x2]


In [ ]:
%%writefile /content/implementation/adapters/plate_recognition.py
import os
import sys
from typing import Dict

import cv2
import numpy as np

from pipeline.config import Config
from pipeline.schemas import clamp01


class PlateRecognizerAdapter:
    """Wraps the existing ai/src pipeline (char.pt raw + LPSR-enhanced + TrOCR + grammar fix)."""

    def __init__(self, cfg: Config, use_lpsr=None, use_trocr=None):
        use_lpsr = cfg.use_lpsr if use_lpsr is None else use_lpsr
        use_trocr = cfg.use_trocr if use_trocr is None else use_trocr
        self.cfg = cfg
        cfg.ensure_ai_on_path()
        self._recognizer = None

        # Correct CWD matters: ref_repo's detection.py appends "./yolov5" to sys.path.
        prev = os.getcwd()
        try:
            if str(cfg.ref_repo_dir) not in sys.path:
                sys.path.insert(0, cfg.ref_repo_dir_str)
            os.chdir(cfg.ref_repo_dir_str)
            from src.pipeline import PlateRecognizer

            self._recognizer = PlateRecognizer(use_lpsr=use_lpsr, use_trocr=use_trocr)
        finally:
            os.chdir(prev)

    def recognize(self, plate_bgr: np.ndarray) -> Dict:
        if plate_bgr is None or plate_bgr.size == 0:
            return {"text": "", "confidence": 0.0, "format_valid": False,
                    "score": 0.0, "source": "none"}
        res = self._recognizer.recognize(plate_bgr)
        score = float(res.get("score", 0.0))
        info = res.get("valid", {})
        return {
            "text": str(res.get("final", "")),
            "confidence": clamp01(score / self.cfg.plate_conf_scale),
            "format_valid": bool(info.get("grammar_ok", False)),
            "score": score,
            "source": res.get("source", "none"),
            "raw_char_text": str(res.get("raw_char_text", "")),
            "sr_char_text": str(res.get("sr_char_text", "")),
            "trocr_text": str(res.get("trocr_text", "")),
        }


In [ ]:
%%writefile /content/implementation/adapters/temporal_filter.py
from typing import Dict, List

import os
import sys

from pipeline.config import Config


class TemporalPlateFilter:
    """Wraps ai/src/temporal.TemporalPlateConsensus for plate-level voting across frames."""

    def __init__(self, cfg: Config):
        cfg.ensure_ai_on_path()
        self.cfg = cfg
        self.temp = None

        prev = os.getcwd()
        try:
            if str(cfg.ref_repo_dir) not in sys.path:
                sys.path.insert(0, cfg.ref_repo_dir_str)
            os.chdir(cfg.ref_repo_dir_str)
            from src.temporal import TemporalPlateConsensus

            self.temp = TemporalPlateConsensus(
                max_missed=cfg.temporal_max_missed,
                match_iou=cfg.temporal_match_iou,
                max_center_ratio=cfg.temporal_max_center_ratio,
            )
        finally:
            os.chdir(prev)

    def update(self, plate_dets: List[Dict]) -> List[Dict]:
        """plate_dets: [{bbox:(x1,y1,x2,y2), text, score, source}] -> mutated with track_id + consensus_text."""
        if not plate_dets:
            self.temp.update([])
            return []
        return self.temp.update(plate_dets)


In [ ]:
%%writefile /content/implementation/pipeline/__init__.py


In [ ]:
%%writefile /content/implementation/pipeline/config.py
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


def default_ai_dir() -> Path:
    """Prefer TRAFFIQ_AI_DIR env var, else the repo's ai/ dir next to this file, else a Colab clone path."""
    env = os.environ.get("TRAFFIQ_AI_DIR")
    if env:
        return Path(env).resolve()
    p = Path(__file__).resolve().parent.parent.parent / "ai"
    if p.exists():
        return p.resolve()
    return Path("/content/traffIQ/ai").resolve()


@dataclass
class Config:
    # ---- directories -------------------------------------------------
    ai_dir: Path = field(default_factory=default_ai_dir)
    ref_repo_dir: Path = None  # ai/third_party/ref_repo
    # ---- model weights ----------------------------------------------
    plate_weights: Path = None     # best.pt  (YOLO26m number-plate detector, ultralytics)
    char_weights: Path = None      # char.pt  (yolov5 character detector)
    lpsr_weights: Path = None      # lpsr_best.pth
    char_names: Path = None        # classes/ocr_class_names.txt
    vehicle_weights: Path = None   # object.pt (yolov5 COCO-style detector)
    # ---- recognition toggles ----------------------------------------
    use_lpsr: bool = True
    use_trocr: bool = True
    # ---- detector thresholds ----------------------------------------
    plate_det_conf: float = 0.25
    vehicle_conf: float = 0.25
    vehicle_iou: float = 0.45
    vehicle_size: int = 640
    # ---- tracking ----------------------------------------------------
    tracker_kind: str = "center"                # "center" (default) | "bytetrack"
    center_threshold: float = 35.0              # px distance for center-point tracker
    bytetrack_buffer: int = 30
    bytetrack_match_thresh: float = 0.8
    bytetrack_min_box_area: int = 100
    # ---- speed (two virtual lines) -----------------------------------
    line_a_y: int = 198                         # first line y (px, image space)
    line_b_y: int = 268                         # second line y (px)
    line_offset: int = 6                        # band half-height around each line
    distance_meters: float = 10.0               # real-world distance between the lines
    speed_estimated: bool = True                # calibration-based estimate -> JSON "estimated"
    direction_ab: str = "NORTH"                 # compass string when crossing A->B
    direction_ba: str = "SOUTH"                 # compass string when crossing B->A
    rearm_frames: int = 30                      # min frames between two events for one track id
    prune_missed: int = 45                      # forget speed state after N unseen frames
    # ---- event contract ---------------------------------------------
    camera_id: str = "CAM-007"
    event_type: str = "vehicle_detection"
    event_prefix: str = "evt"
    timestamp_base: Optional[str] = None        # "YYYY-MM-DDTHH:MM:SS" start-of-video, else "now"
    # ---- plate confidence normalization ------------------------------
    # score from plate_logic is mean char conf + grammar/state bonuses,
    # roughly bounded above by plate_conf_scale; we clamp to [0,1] for the contract.
    plate_conf_scale: float = 16.0
    # ---- temporal plate consensus -----------------------------------
    temporal_max_missed: int = 8
    temporal_match_iou: float = 0.15
    temporal_max_center_ratio: float = 0.75
    # ---- io ----------------------------------------------------------
    output_dir: Path = None

    def __post_init__(self):
        ai = Path(self.ai_dir)
        self.ai_dir = ai.resolve()
        if self.ref_repo_dir is None:
            self.ref_repo_dir = (self.ai_dir / "third_party" / "ref_repo").resolve()
        else:
            self.ref_repo_dir = Path(self.ref_repo_dir).resolve()
        if self.plate_weights is None:
            self.plate_weights = (self.ai_dir / "models" / "best.pt").as_posix()
        if self.char_weights is None:
            self.char_weights = (self.ai_dir / "models" / "char.pt").as_posix()
        if self.lpsr_weights is None:
            self.lpsr_weights = (self.ai_dir / "models" / "lpsr_best.pth").as_posix()
        if self.char_names is None:
            self.char_names = (self.ai_dir / "classes" / "ocr_class_names.txt").as_posix()
        if self.vehicle_weights is None:
            self.vehicle_weights = (self.ref_repo_dir / "weights" / "object.pt").as_posix()
        if self.output_dir is None:
            self.output_dir = (Path(self.ai_dir).parent / "implementation" / "outputs").resolve()
        else:
            self.output_dir = Path(self.output_dir).resolve()
        binary_fields = ["plate_weights", "char_weights", "lpsr_weights", "vehicle_weights", "char_names"]
        for name in binary_fields:
            p = Path(getattr(self, name))
            if not p.suffix:
                setattr(self, name, p.as_posix())
        self.ref_repo_dir_str = str(self.ref_repo_dir)
        self.ai_dir_str = str(self.ai_dir)

    def ensure_ai_on_path(self):
        if self.ai_dir_str not in sys.path:
            sys.path.insert(0, self.ai_dir_str)

    def validate(self, require_all=True):
        missing = [p for label, p in [
            ("plate_weights", self.plate_weights),
            ("char_weights", self.char_weights),
            ("lpsr_weights", self.lpsr_weights),
            ("char_names", self.char_names),
            ("vehicle_weights", self.vehicle_weights),
        ] if not Path(p).exists()]
        if missing:
            msg = "Missing model files: " + ", ".join(missing)
            if require_all:
                raise FileNotFoundError(msg)
            print("[WARN]", msg)
        return self


def colab_config(weights_dir="/content/assets", ai_dir="/content/traffIQ/ai",
                 camera_id="CAM-007", **overrides) -> Config:
    """Build a Config assuming the four model files were uploaded to weights_dir."""
    kw = dict(
        ai_dir=ai_dir,
        plate_weights=str(Path(weights_dir) / "best.pt"),
        char_weights=str(Path(weights_dir) / "char.pt"),
        lpsr_weights=str(Path(weights_dir) / "lpsr_best.pth"),
        vehicle_weights=str(Path(weights_dir) / "object.pt"),
        char_names=str(Path(weights_dir) / "ocr_class_names.txt"),
        camera_id=camera_id,
    )
    kw.update(overrides)
    return Config(**kw)


In [ ]:
%%writefile /content/implementation/pipeline/schemas.py
from __future__ import annotations

import json
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Dict, Optional


def clamp01(x: float) -> float:
    return min(1.0, max(0.0, float(x)))


@dataclass
class VehicleInfo:
    type: str
    type_confidence: float = 0.0


@dataclass
class PlateInfo:
    text: str = ""
    confidence: float = 0.0
    format_valid: bool = False


@dataclass
class SpeedInfo:
    value_kmh: float = 0.0
    estimated: bool = True
    direction: str = ""


@dataclass
class DetectionEvent:
    event_id: str
    event_type: str
    camera_id: str
    timestamp: str
    local_track_id: int
    vehicle: VehicleInfo = field(default_factory=VehicleInfo)
    plate: PlateInfo = field(default_factory=PlateInfo)
    speed: SpeedInfo = field(default_factory=SpeedInfo)

    def to_dict(self) -> dict:
        return {
            "event_id": self.event_id,
            "event_type": self.event_type,
            "camera_id": self.camera_id,
            "timestamp": self.timestamp,
            "local_track_id": self.local_track_id,
            "vehicle": {
                "type": self.vehicle.type,
                "type_confidence": round(clamp01(self.vehicle.type_confidence), 4),
            },
            "plate": {
                "text": self.plate.text,
                "confidence": round(clamp01(self.plate.confidence), 4),
                "format_valid": self.plate.format_valid,
            },
            "speed": {
                "value_kmh": round(float(self.speed.value_kmh), 2),
                "estimated": bool(self.speed.estimated),
                "direction": self.speed.direction,
            },
        }

    def to_json(self) -> str:
        return json.dumps(self.to_dict())


class EventIdFactory:
    """Sequential ids shaped like backend example: evt-20260910-000001."""

    def __init__(self, prefix: str = "evt", start: int = 1, day: Optional[datetime] = None):
        self.prefix = prefix or "evt"
        self.next = start
        self.day = day or datetime.utcnow()

    def next_id(self) -> str:
        n = self.next
        self.next += 1
        return f"{self.prefix}-{self.day.strftime('%Y%m%d')}-{n:06d}"


def make_timestamp(start_base: Optional[str], video_time_s: float) -> str:
    """Timestamp string matching the backend example ('YYYY-MM-DDTHH:MM:SS')."""
    if start_base:
        try:
            base = datetime.strptime(start_base, "%Y-%m-%dT%H:%M:%S")
            return (base + timedelta(seconds=video_time_s)).strftime("%Y-%m-%dT%H:%M:%S")
        except ValueError:
            pass
    return datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")


In [ ]:
%%writefile /content/implementation/pipeline/pipeline.py
import os
from typing import List, Optional

import cv2
import numpy as np

from pipeline.config import Config
from pipeline.schemas import (
    DetectionEvent, EventIdFactory, PlateInfo, SpeedInfo, VehicleInfo, make_timestamp,
)


class DetectionPipeline:
    """End-to-end: vehicle detect -> track -> plate detect/recognize -> temporal voting -> speed.

    process_frame() returns the list of events (JSON-shaped dicts) emitted for that frame,
    normally 0 or 1 when a tracked vehicle completes a line crossing with a read plate.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        from adapters.plate_detector import PlateDetector
        from adapters.plate_recognition import PlateRecognizerAdapter
        from adapters.speed_adapter import LineCrossingSpeedEstimator
        from adapters.temporal_filter import TemporalPlateFilter
        from adapters.tracking_adapter import get_tracker
        from adapters.vehicle_detector import VehicleDetector

        self.vehicle = VehicleDetector(cfg)
        self.tracker = get_tracker(cfg)
        self.plate_detector = PlateDetector(cfg)
        self.plate_recognizer = PlateRecognizerAdapter(cfg)
        self.temporal = TemporalPlateFilter(cfg)
        self.speed = LineCrossingSpeedEstimator(cfg)
        self.event_ids = EventIdFactory(prefix=cfg.event_prefix)
        self.frame_number = 0
        # vehicle id -> best plate observation
        self.plate_state = {}
        self.emitted_ids = set()
        self.last_tracks = []

    def reset(self):
        self.tracker.reset()
        self.speed.reset()
        self.plate_state.clear()
        self.emitted_ids.clear()
        self.frame_number = 0

    def _associate_plate_to_vehicle(self, tracks, plate_box):
        cx = (plate_box[0] + plate_box[2]) / 2.0
        cy = (plate_box[1] + plate_box[3]) / 2.0
        best = None
        best_area = 1e18
        for t in tracks:
            if t.x1 <= cx <= t.x2 and t.y1 <= cy <= t.y2:
                area = (t.x2 - t.x1) * (t.y2 - t.y1)
                if area < best_area:
                    best_area = area
                    best = t
        return best

    def process_frame(self, frame: np.ndarray, fps: float = 25.0) -> List[dict]:
        self.frame_number += 1
        events = []

        vehicles = self.vehicle.detect(frame)
        tracks = self.tracker.update(vehicles)
        self.last_tracks = tracks

        plates = self.plate_detector.detect(frame)
        plate_dets = []
        for p in plates:
            crop = self.plate_detector.crop(frame, p["bbox"])
            if crop.size == 0:
                continue
            rec = self.plate_recognizer.recognize(crop)
            plate_dets.append({**p, **rec})
        plate_dets = self.temporal.update(plate_dets)

        new_plate_state = {}
        for det in plate_dets:
            text = det.get("consensus_text") or det.get("text") or ""
            veh = self._associate_plate_to_vehicle(tracks, det["bbox"])
            if veh is None:
                continue
            key = veh.id
            prev = self.plate_state.get(key)
            conf = float(det.get("confidence", 0.0))
            if prev is not None and prev["skip"]:
                continue
            if text:
                new_plate_state[key] = {
                    "text": text,
                    "confidence": conf,
                    "format_valid": bool(det.get("format_valid", False)),
                    "skip": False,
                }
            elif prev is not None and prev["text"]:
                new_plate_state[key] = prev
        for k, v in new_plate_state.items():
            self.plate_state[k] = v

        for t in tracks:
            sp = self.speed.update(t, self.frame_number, fps)
            if sp is None:
                continue
            key = t.id
            if key in self.emitted_ids:
                continue
            plate = self.plate_state.get(key)
            plate_text = (plate or {}).get("text", "")
            if not plate_text:
                print(f"[EVENT-SKIP] vehicle {key} crossed lines but no plate read yet.")
                continue
            video_time = self.frame_number / fps if fps and fps > 0 else 0.0
            event = DetectionEvent(
                event_id=self.event_ids.next_id(),
                event_type=self.cfg.event_type,
                camera_id=self.cfg.camera_id,
                timestamp=make_timestamp(self.cfg.timestamp_base, video_time),
                local_track_id=key,
                vehicle=VehicleInfo(type=t.type, type_confidence=t.type_confidence),
                plate=PlateInfo(
                    text=plate_text,
                    confidence=float(plate.get("confidence", 0.0)),
                    format_valid=bool(plate.get("format_valid", False)),
                ),
                speed=SpeedInfo(
                    value_kmh=sp["value_kmh"],
                    estimated=sp["estimated"],
                    direction=sp["direction"],
                ),
            )
            self.emitted_ids.add(key)
            events.append(event.to_dict())

        return events


class VideoProcessor:
    """Runs DetectionPipeline over a video file and writes events + annotated output."""

    def __init__(self, cfg: Config, pipeline: Optional[DetectionPipeline] = None):
        self.cfg = cfg
        cfg.output_dir.mkdir(parents=True, exist_ok=True)
        self.pipeline = pipeline or DetectionPipeline(cfg)

    def run(self, video_path: str, output_video: Optional[str] = None,
            jsonl_out: Optional[str] = None, csv_out: Optional[str] = None,
            slice_frames: Optional[int] = None) -> List[dict]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not fps or fps <= 0:
            fps = 20.0

        writer = None
        if output_video:
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(output_video, fourcc, fps, (w, h))

        events = []
        idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            idx += 1
            if slice_frames and idx > slice_frames:
                break
            emitted = self.pipeline.process_frame(frame, fps)
            events.extend(emitted)
            if writer is not None:
                from pipeline.visualization import annotate_frame, draw_lines

                annotated = annotate_frame(frame, self.pipeline, emitted_speed=emitted)
                annotated = draw_lines(annotated, self.cfg)
                writer.write(annotated)
            if idx % 50 == 0:
                print(f"frame {idx}  events_so_far={len(events)}")
        cap.release()
        if writer is not None:
            writer.release()

        if jsonl_out:
            self._write_jsonl(events, jsonl_out)
        if csv_out:
            self._write_csv(events, csv_out)
        return events

    @staticmethod
    def _write_jsonl(events: List[dict], path: str):
        import json as _json

        with open(path, "w", encoding="utf-8") as fh:
            for e in events:
                fh.write(_json.dumps(e) + "\n")

    @staticmethod
    def _write_csv(events: List[dict], path: str):
        import csv as _csv

        rows = []
        for e in events:
            rows.append({
                "event_id": e["event_id"], "event_type": e["event_type"],
                "camera_id": e["camera_id"], "timestamp": e["timestamp"],
                "local_track_id": e["local_track_id"],
                "vehicle_type": e["vehicle"]["type"],
                "vehicle_conf": e["vehicle"]["type_confidence"],
                "plate_text": e["plate"]["text"],
                "plate_conf": e["plate"]["confidence"],
                "plate_format_valid": e["plate"]["format_valid"],
                "speed_kmh": e["speed"]["value_kmh"],
                "speed_estimated": e["speed"]["estimated"],
                "speed_direction": e["speed"]["direction"],
            })
        with open(path, "w", newline="", encoding="utf-8") as fh:
            writer = _csv.DictWriter(fh, fieldnames=list(rows[0]) if rows else [
                "event_id", "event_type", "camera_id", "timestamp", "local_track_id",
                "vehicle_type", "vehicle_conf", "plate_text", "plate_conf",
                "plate_format_valid", "speed_kmh", "speed_estimated", "speed_direction"])
            writer.writeheader()
            if rows:
                writer.writerows(rows)


In [ ]:
%%writefile /content/implementation/pipeline/visualization.py
import cv2
import numpy as np

from pipeline.config import Config
from pipeline.schemas import DetectionEvent


def draw_lines(frame: np.ndarray, cfg: Config) -> np.ndarray:
    a, b = int(cfg.line_a_y), int(cfg.line_b_y)
    h, w = frame.shape[:2]
    cv2.line(frame, (0, a), (w, a), (0, 0, 255), 2)
    cv2.line(frame, (0, b), (w, b), (255, 0, 0), 2)
    cv2.putText(frame, "Line A", (10, max(16, a - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    cv2.putText(frame, "Line B", (10, max(16, b - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    return frame


def annotate_frame(frame: np.ndarray, pipeline, emitted_speed=None) -> np.ndarray:
    annotated = frame.copy()
    speed_by_id = {}
    if emitted_speed:
        for e in emitted_speed:
            speed_by_id[int(e["local_track_id"])] = e

    for t in pipeline.last_tracks:
        cv2.rectangle(annotated, (t.x1, t.y1), (t.x2, t.y2), (0, 255, 0), 2)
        label = f"#{t.id} {t.type}"
        cv2.putText(annotated, label, (t.x1, max(20, t.y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        plate = pipeline.plate_state.get(t.id)
        if plate and plate.get("text"):
            cv2.putText(annotated, plate["text"], (t.x1, t.y2 + 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        if t.id in speed_by_id:
            sp = speed_by_id[t.id]["speed"]
            text = f"{sp['value_kmh']} km/h {sp['direction']}"
            cv2.putText(annotated, text, (t.x1, t.y2 + 36),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    return annotated


def write_events_summary(events, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    jsonl = str(output_dir / "events.jsonl")
    csv = str(output_dir / "events.csv")
    from pipeline.pipeline import VideoProcessor

    VideoProcessor._write_jsonl(events, jsonl)
    VideoProcessor._write_csv(events, csv)
    return jsonl, csv


## 5. Import smoke test


In [ ]:
from pipeline.config import Config
from pipeline.schemas import DetectionEvent, VehicleInfo, PlateInfo, SpeedInfo, EventIdFactory
from adapters.tracking_adapter import CenterPointTracker, TrackedBox
from adapters.speed_adapter import LineCrossingSpeedEstimator
import json

cfg = Config()
tracker = CenterPointTracker(35.0)
a = tracker.update([(10, 10, 60, 60)])
b = tracker.update([(12, 12, 62, 62)])
assert a[0].id == b[0].id, "tracker lost the ID"

speed = LineCrossingSpeedEstimator(cfg)
speed.update(TrackedBox(0, 0, 50, 50, 1, 25, cfg.line_a_y), 100, 25.0)
ev = speed.update(TrackedBox(0, 0, 50, 50, 1, 25, cfg.line_b_y), 108, 25.0)
assert ev is not None and abs(ev["value_kmh"] - 112.5) < 1.0, ev

evt = DetectionEvent(
    event_id=EventIdFactory().next_id(), event_type="vehicle_detection",
    camera_id="CAM-007", timestamp="2026-09-10T20:45:30", local_track_id=42,
    vehicle=VehicleInfo("car", 0.96), plate=PlateInfo("WB12AB1234", 0.91, True),
    speed=SpeedInfo(47.5, True, "NORTH"))
print("SMOKE OK\n", json.dumps(evt.to_dict(), indent=2))


## Next

Open `01_component_tests.ipynb`, then `02_plate_recognition_pipeline.ipynb`, then `03_full_vehicle_anpr_pipeline.ipynb`.
